In [0]:

from pyspark.sql import functions as F

df = spark.table("workspace.default.silver_performance")

# Number of records and distinct indicators by year
year_summary = (
    df.groupBy("FinancialYear")
      .agg(
          F.count("*").alias("Records"),
          F.countDistinct("Council").alias("Councils"),
          F.countDistinct("ID").alias("DistinctIndicators"),
          F.sum(
              F.when(F.col("Result").isNull(), 1).otherwise(0)
          ).alias("MissingResults")
      )
      .orderBy("FinancialYear")
)

display(year_summary)

# Identify indicators that do not exist in all five years
indicator_coverage = (
    df.groupBy("ID", "Description")
      .agg(
          F.countDistinct("FinancialYear").alias("YearsAvailable"),
          F.collect_set("FinancialYear").alias("AvailableYears"),
          F.countDistinct("Council").alias("CouncilsCovered")
      )
)

display(
    indicator_coverage
    .filter(F.col("YearsAvailable") < 5)
    .orderBy("ID", "Description")
)


FinancialYear,Records,Councils,DistinctIndicators,MissingResults
2020-21,4582,79,58,43
2021-22,4582,79,58,99
2022-23,4582,79,58,0
2023-24,4661,79,59,0
2024-25,4661,79,59,0


ID,Description,YearsAvailable,AvailableYears,CouncilsCovered
AF2,Health inspections of aquatic facilities,3,"List(2022-23, 2023-24, 2024-25)",79
AF2,Number of inspections by an authorised officer carried out per Council aquatic facility,2,"List(2020-21, 2021-22)",79
AF6,Number of visits to aquatic facilities per head of municipal population,2,"List(2020-21, 2021-22)",79
AF6,Utilisation of aquatic facilities,3,"List(2022-23, 2023-24, 2024-25)",79
AF7,Cost of aquatic facilities,3,"List(2022-23, 2023-24, 2024-25)",79
AF7,Direct cost to council (less any income received) of providing aquatic facilities per visit. New indicator from July 2019. Please refer to retired indicators link for previous results.,2,"List(2020-21, 2021-22)",79
AM1,Average number of days it has taken for council to action animal management requests,2,"List(2020-21, 2021-22)",79
AM1,Time taken to action animal management requests,3,"List(2022-23, 2023-24, 2024-25)",79
AM2,Animals reclaimed,3,"List(2022-23, 2023-24, 2024-25)",79
AM2,Percentage of all collected cats and dogs which have been reclaimed,2,"List(2020-21, 2021-22)",79


In [0]:
from pyspark.sql import functions as F

display(
    df.filter(F.col("Result").isNull())
      .groupBy("FinancialYear", "ID", "Description")
      .agg(F.count("*").alias("MissingCount"))
      .orderBy("FinancialYear", F.desc("MissingCount"))
)